In [1]:
!pip install langgraph langchain langchain-openai openai python-dotenv

     -------------------------------------- 160.9/160.9 kB 2.4 MB/s eta 0:00:00
     -------------------------------------- 111.7/111.7 kB 6.3 MB/s eta 0:00:00
     ---------------------------------------- 87.2/87.2 kB 5.1 MB/s eta 0:00:00
     ---------------------------------------- 1.1/1.1 MB 7.9 MB/s eta 0:00:00
  Using cached python_dotenv-1.2.2-py3-none-any.whl (22 kB)
     ------------------------------------- 502.7/502.7 kB 10.7 MB/s eta 0:00:00
     ---------------------------------------- 50.5/50.5 kB ? eta 0:00:00
     ---------------------------------------- 90.5/90.5 kB 5.0 MB/s eta 0:00:00
  Using cached pydantic-2.12.5-py3-none-any.whl (463 kB)
     -------------------------------------- 879.4/879.4 kB 8.0 MB/s eta 0:00:00
  Using cached anyio-4.12.1-py3-none-any.whl (113 kB)
  Using cached distro-1.9.0-py3-none-any.whl (20 kB)
  Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
     -------------------------------------- 204.6/204.6 kB 6.1 MB/s eta 0:00:00
  Using cach


[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!sudo apt-get install python3-dev graphviz libgraphviz-dev pkg-config
!pip install graphviz
!pip install pygraphviz

Sudo is disabled on this machine. To enable it, go to the ]8;;ms-settings:developers\Developer Settings page]8;;\ in the Settings app


     ---------------------------------------- 47.3/47.3 kB 1.2 MB/s eta 0:00:00



[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


     -------------------------------------- 106.0/106.0 kB 1.5 MB/s eta 0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
Failed to build pygraphviz


  error: subprocess-exited-with-error
  
  × Building wheel for pygraphviz (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [94 lines of output]
      C:\Users\Divyanshu\AppData\Local\Temp\pip-build-env-z6wwdl7o\overlay\Lib\site-packages\setuptools\config\_apply_pyprojecttoml.py:82: SetuptoolsDeprecationWarning: `project.license` as a TOML table is deprecated
      !!
      
              ********************************************************************************
              Please use a simple string containing a SPDX expression for `project.license`. You can also use `project.license-files`. (Both options available on setuptools>=77.0.0).
      
              By 2027-Feb-18, you need to update your project and remove deprecated calls
              or your builds will no longer be supported.
      
              See https://packaging.python.org/en/latest/guides/writing-pyproject-toml/#license for details.
              ******************************************

In [8]:
!pip install langchain-groq rich

  Using cached rich-14.3.3-py3-none-any.whl (310 kB)
  Using cached markdown_it_py-4.0.0-py3-none-any.whl (87 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl (10.0 kB)



[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
# Utilities
import operator
from functools import reduce
from typing import Annotated, List, Dict, TypedDict, Literal, Optional, Callable, Set, Tuple, Any, Union, TypeVar
from datetime import datetime, timezone, timedelta
import asyncio
from pydantic import BaseModel, Field
from operator import add
from IPython.display import Image, display
import json
import re
import os
# Core imports
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage, BaseMessage
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langgraph.graph import StateGraph, END, START


# Pretty Markdown Output
from rich.console import Console
from rich.markdown import Markdown
from rich.panel import Panel
from rich.text import Text
from rich import box
from rich.style import Style

## State Definition
Define the AcademicState class to hold the workflow's state.


In [10]:
T = TypeVar('T')

def dict_reducer(dict1: Dict[str, Any], dict2: Dict[str, Any]) -> Dict[str, Any]:
    """
    Merge two dictionaries recursively

    Example:
    dict1 = {"a": {"x": 1}, "b": 2}
    dict2 = {"a": {"y": 2}, "c": 3}
    result = {"a": {"x": 1, "y": 2}, "b": 2, "c": 3}
    """
    merged = dict1.copy()
    for key, value in dict2.items():
        if key in merged and isinstance(merged[key], dict) and isinstance(value, dict):
            merged[key] = dict_reducer(merged[key], value)
        else:
            merged[key] = value
    return merged
     

In [11]:
class AcademicState(TypedDict):
    """Master state container for the academic assistance system"""
    #  messages: Annotated[List[BaseMessage], add]   # Conversation history
    #  profile: dict                                 # Student information
    #  calendar: dict                                # Scheduled events
    #  tasks: dict                                   # To-do items and assignments
    #  results: Dict[str, Any]                       # Operation outputs
    messages: Annotated[List[BaseMessage], add]   # Conversation history
    profile: Annotated[Dict, dict_reducer]                 # Student information
    calendar: Annotated[Dict, dict_reducer]                # Scheduled events
    tasks: Annotated[Dict, dict_reducer]                   # To-do items and assignments
    results: Annotated[Dict[str, Any], dict_reducer]       # Operation outputs
     

## LLM Initialization

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage
from typing import List, Dict, Optional

class LLMConfig:
    """Configuration settings for the LLM (Groq)."""
    model: str = "llama-3.3-70b-versatile"
    max_tokens: int = 1024
    default_temp: float = 0.5


class GroqLLM:
    """
    Async wrapper around Groq LLM using LangChain.
    Acts as a drop-in replacement for AsyncOpenAI-based Nemotron client.
    """

    def __init__(self, api_key: str):
        self.config = LLMConfig()
        self.client = ChatGroq(
            groq_api_key=api_key,
            model=self.config.model,
            temperature=self.config.default_temp,
            max_tokens=self.config.max_tokens,
        )
        self._is_authenticated = False

    async def check_auth(self) -> bool:
        try:
            await self.agenerate(
                [{"role": "user", "content": "ping"}],
                temperature=0.1,
            )
            self._is_authenticated = True
            return True
        except Exception as e:
            print(f"❌ Authentication failed: {e}")
            return False

    async def agenerate(
        self,
        messages: List[Dict],
        temperature: Optional[float] = None,
    ) -> str:
        """
        Generate text asynchronously using Groq LLM.
        """

        lc_messages = []
        for m in messages:
            if m["role"] == "system":
                lc_messages.append(SystemMessage(content=m["content"]))
            else:
                lc_messages.append(HumanMessage(content=m["content"]))

        response = await self.client.ainvoke(
            lc_messages,
            temperature=temperature or self.config.default_temp,
        )

        return response.content